In [0]:
dbutils.widgets.removeAll()

In [0]:

from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:

dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsssmartdata2698")

In [0]:

container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/olist_order_payments_dataset.csv"

In [0]:
df_payments = spark.read.option("header", True)\
                        .option("inferSchema", True)\
                        .csv(ruta)

In [0]:
payments_schema = StructType(fields=[
    StructField("order_id", StringType(), True),
    StructField("payment_sequential", IntegerType(), True),
    StructField("payment_type", StringType(), True),
    StructField("payment_installments", IntegerType(), True),
    StructField("payment_value", DoubleType(), True)
])


In [0]:
df_payments_final = spark.read\
.option("header", True)\
.schema(payments_schema)\
.csv(ruta)

In [0]:
payments_selected_df = df_payments_final.select(
    col("order_id"),
    col("payment_sequential"),
    col("payment_type"),
    col("payment_installments"),
    col("payment_value")
)


In [0]:
payments_renamed_df = payments_selected_df

In [0]:
payments_final_df = payments_renamed_df.withColumn("ingestion_date", current_timestamp())

In [0]:
payments_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.payments_raw")